<a href="https://colab.research.google.com/github/CarloQuick/anodized_mistral/blob/main/mistral_hack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mistralai pandas wandb unsloth trl
from google.colab import userdata
from google.colab import files
import wandb
import json
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.3/509.3 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
mistral_key = userdata.get('MISTRAL_API_KEY')
wandb_key = userdata.get('WANDB_API_KEY')
hf_token = userdata.get('HF_TOKEN')

In [ ]:
# Fine Tune
wandb.login(key=wandb_key)
wandb.init(project="anodized-mistral")

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# Load training data
examples = []
with open("training.jsonl", "r") as f:
    for line in f:
        examples.append(json.loads(line))

def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": text}

dataset = Dataset.from_list(examples).map(format_chat)

# Load eval data
eval_examples = []
with open("eval.jsonl", "r") as f:
    for line in f:
        eval_examples.append(json.loads(line))

eval_dataset = Dataset.from_list(eval_examples).map(format_chat)

# Train with eval
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        num_train_epochs=3,
        learning_rate=2e-4,
        output_dir="outputs",
        report_to="wandb",
        logging_steps=1,
        eval_strategy="epoch",
    ),
)

trainer.train()
wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: carlo-quick (mistral-hackathon-2026) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

...

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12 | Num Epochs = 3 | Total steps = 36
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 13,631,488 of 7,261,655,040 (0.19% trained)


Epoch,Training Loss,Validation Loss
1,1.550600,2.203526
2,0.883800,1.583352
3,0.853800,1.502187


eval/loss,█▂▁
eval/runtime,█▁▁
eval/samples_per_second,▁██
eval/steps_per_second,▁██
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,▅ ▃▇█▂▆▃▂▄▂▂▂▃▂▃▃▁▂▁▂▂▂▁▂▂▂▁▂▂▁▁
train/learning_rate,███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
train/loss,▃▅▄▆▅▅▃█▃▃▅▂▂▂▃▂▂▂▂▃▂▁▃▁▁▂▁▁▂▁▁▁▃▂▁▁
eval/loss,1.50219
eval/runtime,0.9962


In [ ]:
# model.save_pretrained("anodized-lora")
# tokenizer.save_pretrained("anodized-lora")

from huggingface_hub import login


login(token=hf_token)
model.push_to_hub("cQuick/anodized-mistral-lora")
tokenizer.push_to_hub("cQuick/anodized-mistral-lora")

README.md:   0%|          | 0.00/574 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  560kB / 54.6MB            

Saved model to https://huggingface.co/cQuick/anodized-mistral-lora


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pf1326j4p/tokenizer.model: 100%|##########|  587kB /  587kB            

In [ ]:
# All training data available on https://github.com/mkovaxx/anodized
hold_outs = [
{
    "input": """fn safe_index(slice: &[i32], index: usize) -> i32 {
    slice[index]
}""",
    "output": """#[spec(
    requires: index < slice.len(),
)]"""
},
{
    "input": """fn safe_subtract(a: u32, b: u32) -> u32 {
    a - b
}""",
    "output": """#[spec(
    requires: a >= b,
)]"""
},
{
    "input": """fn reciprocal(x: f64) -> f64 {
    1.0 / x
}""",
    "output": """#[spec(
    requires: [
        x != 0.0,
        x.is_finite(),
    ],
    ensures: output.is_finite(),
)]"""
},
{
"input":
"""
trait TestTrait {
    fn current(&self) -> i32;

    fn add_to(&self, x: i32) -> i32;
}
""",

    "output":
"""
#[spec(
    requires: x > 0,
    captures: self.current() as old_val,
    ensures: *output > old_val,
)]
"""
},
]

In [ ]:
# Load base model WITHOUT LoRA
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

system_prompt = "Generate an anodized #[spec] annotation for the given Rust code. Include preconditions (requires), postconditions (ensures), and invariants where appropriate. Do not wrap in markdown, code fences, or any formatting."

for h in hold_outs:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": h["input"]}
    ]
    inputs = base_tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
    outputs = base_model.generate(inputs, max_new_tokens=256)
    result = base_tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print("GOT:", result)
    print("---")

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


GOT: Here is an example of an annotated version of the given Rust code using the #[spec] annotation. I've included preconditions, postconditions, and invariants where appropriate.

```rust
#![feature(plugin)]
#![plugin(spec)]

use spec::prelude::*;

#[spec]
fn safe_index(slice: &[i32], index: usize) -> i32 {
    requires slice: is_slice_of<i32>();
    requires index: is_valid_index(slice.len());

    let result = slice[index];

    ensures result: is_some_value();
    ensures slice: is_slice_of<i32>();
    ensures index: is_valid_index(slice.len());

    result.unwrap()
}
```

In this example, I've used the `is_slice_of<T>` and `is_valid_index(len: usize)` traits from the `spec` crate to define preconditions for the function. The `is_some_value()` trait is used in the post
---
GOT: Here is an example of an annotated Rust function using the #[spec] attribute for the `safe_subtract` function. I've included preconditions, postconditions, and invariants.

```rust
#![feature(specs)]

#[spec

In [ ]:
# Holds with trained model
FastLanguageModel.for_inference(model)

system_prompt = "Generate an anodized #[spec] annotation for the given Rust code. Include preconditions (requires), postconditions (ensures), and invariants where appropriate. Do not wrap in markdown, code fences, or any formatting."

for h in hold_outs:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": h["input"]}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
    outputs = model.generate(inputs, max_new_tokens=256)
    result = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print("INPUT:", h["input"][:60])
    print("EXPECTED:", h["output"])
    print("GOT:", result)
    print("---")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


INPUT: fn safe_index(slice: &[i32], index: usize) -> i32 {
    slic
EXPECTED: #[spec(
    requires: index < slice.len(),
)]
GOT: #[spec(
    requires: slice.len() > index,
    ensures: *slice.get(index).unwrap() == *output,
    maintains: *slice.get(0) <= *slice.get(1) <= *slice.get(2) <= *slice.get(3) <= *slice.get(4) <= *slice.get(5) <= *slice.get(6) <= *slice.get(7) <= *slice.get(8) <= *slice.get(9),
)]
---
INPUT: fn safe_subtract(a: u32, b: u32) -> u32 {
    a - b
}
EXPECTED: #[spec(
    requires: a >= b,
)]
GOT: #[spec(
    requires: a >= b,
    ensures: *output <= a && *output >= b,
    maintains: *output == a - b,
)]
---
INPUT: fn reciprocal(x: f64) -> f64 {
    1.0 / x
}
EXPECTED: #[spec(
    requires: [
        x != 0.0,
        x.is_finite(),
    ],
    ensures: output.is_finite(),
)]
GOT: #[spec(
    requires: x > 0.0,
    ensures: *output == 1.0 / x,
    maintains: *output * x == 1.0,
)]
---
INPUT: 
trait TestTrait {
    fn current(&self) -> i32;

    fn add
EXPECTED: 
#[sp